# Lab 3 - Bayes and kNN (forest vs urban)

I classify pixels from a mixed satellite-ish image as forest or urban.

Training:
- `forest.png` — mostly forest pixels
- `urban.png` — mostly urban pixels

Test:
- `mixed.png` — both classes together

Colors on maps: urban = red, forest = blue


## Imports and loading images


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

%matplotlib inline
np.random.seed(7)

forest = np.array(Image.open('forest.png').convert('RGB'))
urban = np.array(Image.open('urban.png').convert('RGB'))
mixed = np.array(Image.open('mixed.png').convert('RGB'))

print(forest.shape, urban.shape, mixed.shape)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(forest)
plt.title('forest train')
plt.axis('off')
plt.subplot(1, 3, 2)
plt.imshow(urban)
plt.title('urban train')
plt.axis('off')
plt.subplot(1, 3, 3)
plt.imshow(mixed)
plt.title('mixed test')
plt.axis('off')
plt.show()

Xf = forest.reshape(-1, 3).astype(float)
Xu = urban.reshape(-1, 3).astype(float)
Xt = mixed.reshape(-1, 3).astype(float)
print('pixels forest/urban/test:', len(Xf), len(Xu), len(Xt))


## Histograms of training data

I look at R, G, B separately for forest and urban.


In [ ]:
names = ['R', 'G', 'B']
fig, axes = plt.subplots(2, 3, figsize=(11, 5))
for i in range(3):
    axes[0, i].hist(Xf[:, i], bins=40, density=True, color='green', alpha=0.8)
    axes[0, i].set_title('forest ' + names[i])
    axes[1, i].hist(Xu[:, i], bins=40, density=True, color='gray', alpha=0.8)
    axes[1, i].set_title('urban ' + names[i])
plt.tight_layout()
plt.show()


Forest looks greener / a bit darker and the values are more concentrated.
Urban is brighter and more spread (roofs, roads, shadows...).


## Bayes classifier from scratch

Idea:
1. each pixel is a RGB vector
2. for each class I fit a Gaussian: mean mu and covariance Sigma
3. equal priors 0.5 / 0.5
4. pick class with bigger log posterior


In [ ]:
def fit_gauss(X):
    mu = X.mean(axis=0)
    Sigma = np.cov(X, rowvar=False) + 1e-6 * np.eye(3)
    return mu, Sigma


def logpdf_gauss(X, mu, Sigma):
    d = X.shape[1]
    diff = X - mu
    inv = np.linalg.inv(Sigma)
    sign, logdet = np.linalg.slogdet(Sigma)
    q = np.sum(diff @ inv * diff, axis=1)
    return -0.5 * (d * np.log(2 * np.pi) + logdet + q)


mu_f, S_f = fit_gauss(Xf)
mu_u, S_u = fit_gauss(Xu)

print('mu forest =', np.round(mu_f, 2))
print('mu urban  =', np.round(mu_u, 2))
print('Sigma forest =')
print(np.round(S_f, 2))
print('Sigma urban =')
print(np.round(S_u, 2))


In [ ]:
prior = 0.5
lf = logpdf_gauss(Xt, mu_f, S_f) + np.log(prior)
lu = logpdf_gauss(Xt, mu_u, S_u) + np.log(prior)

bayes_labels = (lu > lf).astype(np.uint8).reshape(mixed.shape[0], mixed.shape[1])

bayes_map = np.zeros_like(mixed)
bayes_map[bayes_labels == 1] = [220, 40, 40]
bayes_map[bayes_labels == 0] = [40, 80, 220]

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(mixed)
plt.title('mixed')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(bayes_map)
plt.title('Bayes (red=urban, blue=forest)')
plt.axis('off')
plt.show()

print('urban % =', round(100 * bayes_labels.mean(), 1))
print('forest % =', round(100 * (1 - bayes_labels.mean()), 1))


### Bayes notes

mu is the average RGB of the class. Sigma says how much the colors vary and how channels move together.

Forest mu has relatively more green and lower overall brightness than urban in my run.
Urban Sigma is usually larger because city materials are more mixed.

Equal priors are fine here because both training images are similar size.


## kNN from scratch

Steps:
1. take some random training pixels from forest and urban
2. for each test pixel find k nearest in RGB space (euclidean)
3. majority vote
4. try k = 1, 3, 5


In [ ]:
from scipy.spatial.distance import cdist

n = 1200
idf = np.random.choice(len(Xf), n, replace=False)
idu = np.random.choice(len(Xu), n, replace=False)

Xtr = np.vstack([Xf[idf], Xu[idu]])
ytr = np.hstack([np.zeros(n, dtype=np.uint8), np.ones(n, dtype=np.uint8)])

print('train size', Xtr.shape)


def knn(Xtr, ytr, Xq, k):
    D = cdist(Xq, Xtr, metric='euclidean')
    nn = np.argpartition(D, kth=k-1, axis=1)[:, :k]
    votes = ytr[nn]
    return (votes.sum(axis=1) * 2 >= k).astype(np.uint8)


knn_maps = {}
for k in [1, 3, 5]:
    lab = knn(Xtr, ytr, Xt, k).reshape(mixed.shape[0], mixed.shape[1])
    knn_maps[k] = lab
    m = np.zeros_like(mixed)
    m[lab == 1] = [220, 40, 40]
    m[lab == 0] = [40, 80, 220]
    print('k=', k, 'urban % =', round(100 * lab.mean(), 1))


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
axes[0].imshow(mixed)
axes[0].set_title('mixed')
axes[0].axis('off')

for i, k in enumerate([1, 3, 5]):
    m = np.zeros_like(mixed)
    m[knn_maps[k] == 1] = [220, 40, 40]
    m[knn_maps[k] == 0] = [40, 80, 220]
    axes[i+1].imshow(m)
    axes[i+1].set_title('kNN k=' + str(k))
    axes[i+1].axis('off')

plt.tight_layout()
plt.show()


### kNN notes

k=1 follows only one neighbour so the map is more noisy (salt and pepper).

k=3 and k=5 look smoother. Small wrong spots disappear more often.

Too big k could blur thin edges, but here 3 or 5 looks ok.


## Quick comparison


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].imshow(mixed)
axes[0].set_title('original')
axes[1].imshow(bayes_map)
axes[1].set_title('Bayes')

m5 = np.zeros_like(mixed)
m5[knn_maps[5] == 1] = [220, 40, 40]
m5[knn_maps[5] == 0] = [40, 80, 220]
axes[2].imshow(m5)
axes[2].set_title('kNN k=5')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

for name, lab in [('Bayes', bayes_labels), ('kNN1', knn_maps[1]), ('kNN3', knn_maps[3]), ('kNN5', knn_maps[5])]:
    agree = 100 * np.mean(lab == bayes_labels)
    print(name, 'urban%', round(100*lab.mean(),1), 'agree with Bayes%', round(agree,1))


## Summary

Bayes: I estimated Gaussian PDFs from pure forest/urban pixels and classified mixed by higher posterior.

kNN: same RGB features, vote of nearest training pixels. k=1 is noisier, k=3/5 smoother.

Both methods find roughly the same pattern: buildings red, trees/grass blue.
